# Create Folds

Stratified 5-fold CV split at the **clip** level, stratified on each clip's dominant action.

- No clip is split across folds (prevents temporal leakage between near-duplicate CCTV frames).
- Each fold gets a proportional share of every action class — important because `using_laptop` only has 32 segments total.
- Seeded so all three models (ST-GATv2, LSTM, MLP) train on identical splits.
- Output: `folds.json` next to the dataset. Reload it from any training notebook.

## Imports

In [2]:
import sys
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedKFold

In [3]:
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [4]:
from src.utils.config.config import Config

## Config

In [5]:
SEED = 42
K = 5

config_loader = Config()
cfg = config_loader.load_config()

dataset_root = cfg.project_root / cfg.paths.dataset / "CafeV1"
clips_root   = dataset_root / "Clips"
folds_path   = dataset_root / "folds.json"

with open(dataset_root / "action_classes.json", "r") as f:
    ACTION_NAME_TO_ID = json.load(f)

ACTION_ID_TO_NAME = {v: k for k, v in ACTION_NAME_TO_ID.items()}

## Collect clips and their dominant action

Each clip gets one stratification label = the action with the most segments in that clip.
Clips without `hoi-anns.json` (not yet HOI-annotated) are skipped.

In [6]:
def dominant_action(hoi_path):
    with open(hoi_path, "r") as f:
        data = json.load(f)
    counts = Counter(seg["action_id"] for seg in data.get("annotations", []))
    if not counts:
        return None
    # ties broken by lowest action_id (deterministic)
    top = max(counts.values())
    return min(aid for aid, n in counts.items() if n == top)

def action_distribution(hoi_path):
    with open(hoi_path, "r") as f:
        data = json.load(f)
    return Counter(seg["action_id"] for seg in data.get("annotations", []))

clips = []          # list of (vp, clip_id, dominant_action_id)
clip_segments = {}  # (vp, clip_id) -> Counter of action_id -> n
skipped = []

for vp_dir in sorted(clips_root.iterdir(), key=lambda p: int(p.name)):
    if not vp_dir.is_dir():
        continue
    for clip_dir in sorted(vp_dir.iterdir(), key=lambda p: int(p.name)):
        hoi = clip_dir / "hoi-anns.json"
        if not hoi.exists():
            skipped.append(f"vp{vp_dir.name}/clip{clip_dir.name} (no hoi-anns.json)")
            continue
        dist = action_distribution(hoi)
        if not dist:
            skipped.append(f"vp{vp_dir.name}/clip{clip_dir.name} (empty annotations)")
            continue
        dom = max(dist, key=lambda aid: (dist[aid], -aid))
        clips.append((vp_dir.name, clip_dir.name, dom))
        clip_segments[(vp_dir.name, clip_dir.name)] = dist

print(f"Eligible clips: {len(clips)}")
print(f"Skipped: {len(skipped)}")
for s in skipped:
    print(f"  {s}")

print("\nDominant-action distribution across clips:")
dom_counter = Counter(d for _, _, d in clips)
for aid, n in sorted(dom_counter.items()):
    print(f"  {ACTION_ID_TO_NAME[aid]:15s} {n}")

Eligible clips: 126
Skipped: 0

Dominant-action distribution across clips:
  idle            116
  using_phone     5
  reading         5


## Build the folds

In [7]:
clip_keys   = [(vp, cid) for vp, cid, _ in clips]
strat_labels = [d for _, _, d in clips]

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

assignments = {}                  # "vp/cid" -> fold_idx
folds = [[] for _ in range(K)]    # fold_idx -> list of "vp/cid"

for fold_idx, (_, val_idx) in enumerate(skf.split(clip_keys, strat_labels)):
    for i in val_idx:
        vp, cid = clip_keys[i]
        key = f"{vp}/{cid}"
        assignments[key] = fold_idx
        folds[fold_idx].append(key)

# Sort each fold's clip list for stable output
for f in folds:
    f.sort(key=lambda s: (int(s.split("/")[0]), int(s.split("/")[1])))

print(f"Built {K} folds over {len(clip_keys)} clips with seed={SEED}.\n")
for i, f in enumerate(folds):
    print(f"Fold {i}: {len(f)} clips")

Built 5 folds over 126 clips with seed=42.

Fold 0: 26 clips
Fold 1: 25 clips
Fold 2: 25 clips
Fold 3: 25 clips
Fold 4: 25 clips


## Fold composition

Per-fold action *segment* counts (not just dominant-action of clips — the real distribution the models will see).
The rare class to watch is `using_laptop`.

In [8]:
rows = {}
for fold_idx, clip_keys_in_fold in enumerate(folds):
    seg_counts = Counter()
    for key in clip_keys_in_fold:
        vp, cid = key.split("/")
        seg_counts.update(clip_segments[(vp, cid)])
    rows[f"fold_{fold_idx}"] = [seg_counts.get(aid, 0) for aid in sorted(ACTION_ID_TO_NAME)]

df = pd.DataFrame(rows, index=[ACTION_ID_TO_NAME[aid] for aid in sorted(ACTION_ID_TO_NAME)])
df["TOTAL"] = df.sum(axis=1)
df.loc["TOTAL"] = df.sum(axis=0)
df

,fold_0,fold_1,fold_2,fold_3,fold_4,TOTAL
idle,220,192,222,209,211,1054
using_laptop,7,9,1,5,10,32
using_phone,122,91,115,110,122,560
reading,55,69,56,51,46,277
TOTAL,404,361,394,375,389,1923


## Save

`folds.json` schema:
```
{
  "seed": 42,
  "k": 5,
  "stratify_key": "dominant_action_per_clip",
  "assignments": { "vp/clip_id": fold_idx, ... },
  "folds":       [ ["vp/clip_id", ...], ... ]   # fold_idx -> val clips
}
```

Training code should treat fold `i` as val and the other K-1 folds as train.

In [9]:
payload = {
    "seed": SEED,
    "k": K,
    "stratify_key": "dominant_action_per_clip",
    "assignments": dict(sorted(assignments.items(), key=lambda kv: (int(kv[0].split("/")[0]), int(kv[0].split("/")[1])))),
    "folds": folds,
}

with open(folds_path, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Wrote {folds_path}")

Wrote C:\Research\Dataset\data\processed\CafeV1\folds.json


## Loader (copy into training notebooks)

```python
import json
from pathlib import Path

def load_folds(folds_path):
    with open(folds_path) as f:
        return json.load(f)

def split(folds_data, val_fold):
    """Returns (train_clips, val_clips) as lists of 'vp/cid' strings."""
    val   = folds_data["folds"][val_fold]
    train = [c for i, f in enumerate(folds_data["folds"]) if i != val_fold for c in f]
    return train, val
```